In [1]:
import cadquery as cq
from jupyter_cadquery import show


def create_triangle_outline(part, outer_thickness=1):
    
    bbox = part.val().BoundingBox()
    
    def outline_line(inp, return_z = True):
        slope = (bbox.zmax - bbox.zmin)/(bbox.xmax - bbox.xmin)
        new_intercept = bbox.zmax - outer_thickness * 2    
        
        if return_z:
            return slope * inp + new_intercept
        else:
            return (inp - new_intercept)/slope
            
    
    inner = (
        cq.Workplane("XZ")
        .polyline([(-outer_thickness, outer_thickness), 
                   (-outer_thickness,outline_line(-outer_thickness)), 
                   (outline_line(outer_thickness, return_z = False), outer_thickness)])
        .close()
        .extrude(-5)
    )
    
    hollow_triangle = part.cut(inner)

    return hollow_triangle



Overwriting auto display for cadquery Workplane and Shape


In [2]:
def get_grid_infill(part, outline, spacing = 5.0, rod_diameter = 1):
    
    width = 70  
    depth = 70   
    thickness = 20  
    
    grid = cq.Workplane("XZ") 
    
    x = -width/2
    while x <= width/2:
        grid = grid.union(
            cq.Workplane("XZ").center(x, 25)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing
    
    z = -depth/2
    while z <= depth/2:
        grid = grid.union(
            cq.Workplane("XZ").center(0, z+25)
            .rect(width, rod_diameter)  
            .extrude(-thickness)         
        )
        z += spacing
    
    grid = grid.rotate((0, 0, 0), (0, 1, 0), 45)
    infill_inside = grid.intersect(part)
    
    result = outline.union(infill_inside)
    return result





In [29]:
def get_finray_infill(part, outline, spacing = 3, rod_diameter = 1):

    width = 70
    depth = 70
    thickness = 20

    grid = cq.Workplane("XZ")

    x = -width/2
    while x <= width/2:
        grid = grid.union(
            cq.Workplane("XZ").center(x, 25)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing

    grid = grid.rotate((0, 0, 0), (0, 1, 0), -60)
    
    infill_inside = grid.intersect(part)
    
    result = outline.union(infill_inside)
    
    return result
    
    
    

In [139]:
import math

def get_triangle_infill(part, outline, spacing = 7, rod_diameter = 1):
    
    width = 60  
    depth = 60  
    thickness = 20  
    
    grid = cq.Workplane("XZ")
    
    x = -width/2
    while x <= width/2:
        grid = grid.union(
            cq.Workplane("XZ").center(x, 3)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing

    grid = grid.rotate((0, 0, 0), (0, 1, 0), -90)

    grid_tri_one = cq.Workplane("XZ")

    x = -width/2
    while x <= width/2:
        grid_tri_one = grid_tri_one.union(
            cq.Workplane("XZ").center(x, 3)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing
        
    grid_tri_one = grid_tri_one.rotate((0, 0, 0), (0, 1, 0), 30)

    grid_tri_two = cq.Workplane("XZ")

    x = -width/2
    while x <= width/2:
        grid_tri_two = grid_tri_two.union(
            cq.Workplane("XZ").center(x, 3)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing
        
    grid_tri_two = grid_tri_two.rotate((0, 0, 0), (0, 1, 0), -30)

    grid_tri = grid_tri_two.union(grid_tri_one)
    grid = grid.union(grid_tri)
    
    grid = grid.rotate((0, 0, 0), (0, 1, 0), -45)

    infill_inside = grid.intersect(part)

    result = outline.union(infill_inside)
    
    return result

In [147]:

part = cq.importers.importStep("STEP_files/GripperForOpt_v2.step")

outline = create_triangle_outline(part, outer_thickness=1)
result = get_triangle_infill(part, outline, spacing = 5, rod_diameter = 0.5 )


show(result)

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…